# Pipeline Medallion — `ecommerce_produtos`

Este notebook implementa a arquitetura **Medallion (Bronze → Silver → Gold)** para a tabela de produtos do e-commerce, utilizando **PySpark no Databricks** com armazenamento no **Azure Data Lake Storage Gen2 (ADLS)** e escrita no formato **Delta Lake**.

---

## 🗂️ Fluxo do Pipeline

```
RAW (CSV no ADLS)
      │
      ▼
  🥉 BRONZE  — Ingestão bruta com metadados de auditoria + validação de contrato
      │
      ▼
  🥈 SILVER  — Deduplicação, tipagem financeira, normalização de booleanos
      │
      ▼
  🥇 GOLD    — KPIs de preço médio semestral + SKUs ativos por categoria/mês
      │
      ▼
  🗄️ SQL SERVER — Exportação dos KPIs para consumo em relatórios/BI
```

---

## 📐 Regras de Negócio Aplicadas (Silver)

| # | Regra | Descrição |
|---|-------|-----------|
| 1 | **Unicidade e deduplicação** | Remoção de registros com `sku` nulo e eliminação de duplicatas pela mesma chave |
| 2 | **Tipagem financeira** | Normalização decimal (vírgula → ponto), cast para `Decimal(10,2)` em `preco_lista`, filtro `> 0` |
| 3 | **Normalização booleana** | Cast de `is_ativo` para boolean com imputação de `False` para nulos via `coalesce` |
| 4 | **Validação de contrato** | Schema enforcement: valida quantidade, nomes e ordem das colunas antes da ingestão |
| 5 | **Auditoria da camada** | Timestamp `silver_processed_at` para rastreamento de quando o registro foi processado |

> ⚠️ A escrita da Silver usa `mode("overwrite")` (não `append`): o DataFrame é recalculado sobre a Bronze inteira a cada execução, então `overwrite` é o que garante `sku` único na tabela física entre execuções (ver seção 3).

---

## 📊 KPIs Gerados (Gold)

| Tabela Gold | Descrição | Granularidade |
|-------------|-----------|---------------|
| `gold_hist_preco_medio_semestre` | Preço médio por categoria/subcategoria (apenas produtos ativos), cruzado com `ecommerce_categorias` | Ano + Semestre + Categoria |
| `gold_kpi_skus_ativos_categoria_mes` | Contagem de SKUs ativos distintos por categoria, por mês | Ano + Mês + Categoria |

---

## 🔧 Dependências e Pré-requisitos

- **Variáveis de ambiente** (arquivo `../env`): `CLIENT_ID`, `TENANT_ID`, `CLIENT_SECRET`, `STORAGE_ACCOUNT_NAME`, `SQL_HOST`, `SQL_DATABASE`, `SQL_USERNAME`, `SQL_PASSWORD`
- **Autenticação no ADLS**: OAuth 2.0 via Service Principal (Client Credentials)
- **Dataset SQL Server**: `squad3`
- **Tabela Silver de outro pipeline**: `ecommerce_categorias` (Delta) — necessária para os dois KPIs Gold (join para `nome_categoria`/`tipo_categoria`); precisa já ter sido processada antes deste notebook

## 1. Setup e Credenciais

Carrega as variáveis de ambiente a partir do arquivo `../env` e configura:
- As opções de autenticação OAuth 2.0 para acesso ao ADLS Gen2 via Service Principal
- Os caminhos base das camadas **Raw** e **Bronze** no Data Lake
- Períodos de execução (`ANO_EXEC`, `MES_EXEC`, `DIA_EXEC`) para rastreamento semanal

> ⚠️ Nenhuma credencial é hardcoded. Todas as chaves sensíveis são lidas exclusivamente via `os.getenv()`.

In [0]:
# ============================================================
# 1. SETUP E CREDENCIAIS (ecommerce_produtos)
# ============================================================

import os
from datetime import date

from dotenv import load_dotenv

from pyspark.sql.functions import (
    current_timestamp,
    year,
    month,
    col,
    count,
    countDistinct,
    avg,
    round,
    when,
    regexp_replace,
    coalesce,
    lit,
    date_format
)


# ============================================================
# 1.1 CARREGAMENTO DO .ENV + FALLBACK PARA JOB PARAMETERS
# ============================================================
# Objetivo:
# - Continuar funcionando manualmente com .env
# - Funcionar via Databricks Job usando Job Parameters
# - Evitar spark.conf.set(), pois no Databricks Free/Serverless
#   algumas configs fs.azure.* podem não estar disponíveis
# ============================================================

try:
    load_dotenv("../env")
    load_dotenv("../.env")
    load_dotenv(".env")
    print("Tentativa de carregamento do .env realizada.")
except Exception as e:
    print(f"Não foi possível carregar .env. Seguindo com fallback. Detalhe: {e}")


def get_config_value(name: str, required: bool = True, default: str = "") -> str:
    """
    Busca uma configuração na seguinte ordem:

    1. Variáveis de ambiente carregadas pelo .env
    2. Databricks Job Parameters, via dbutils.widgets.get()
    3. Valor default, quando informado

    Isso permite que o notebook funcione tanto em execução manual
    quanto em execução agendada pelo Databricks Job.
    """

    value = os.getenv(name)

    if value is None or str(value).strip() == "":
        try:
            value = dbutils.widgets.get(name)
        except Exception:
            value = None

    if value is None or str(value).strip() == "":
        value = default

    value = str(value).strip() if value is not None else ""

    if required and value == "":
        raise ValueError(
            f"Configuração obrigatória não encontrada: {name}. "
            f"Verifique se ela existe no .env ou nos Job Parameters do Databricks."
        )

    return value


# ============================================================
# 1.2 VARIÁVEIS ADLS GEN2
# ============================================================

client_id = get_config_value("CLIENT_ID")
tenant_id = get_config_value("TENANT_ID")
client_secret = get_config_value("CLIENT_SECRET")
storage_account = get_config_value("STORAGE_ACCOUNT_NAME")

# Pelo seu path original, o arquivo está em:
# abfss://raw@storage/batch-data/ecommerce_produtos.csv
source_folder = get_config_value("CONTAINER_NAME", required=False, default="batch-data")

# Containers usados no projeto
raw_container = get_config_value("RAW_CONTAINER", required=False, default="raw")
squad_container = get_config_value("SQUAD_CONTAINER", required=False, default="squad3")


# ============================================================
# 1.3 OPÇÕES DE AUTENTICAÇÃO ADLS GEN2
# ============================================================
# Importante:
# NÃO usar spark.conf.set() aqui.
#
# A estratégia é manter as opções em dicionário e usar nas leituras
# e escritas com .options(**adls_options), caso as células seguintes
# já estejam seguindo esse padrão.
# ============================================================

storage_account_fqdn = f"{storage_account}.dfs.core.windows.net"

adls_options = {
    f"fs.azure.account.auth.type.{storage_account_fqdn}": "OAuth",
    f"fs.azure.account.oauth.provider.type.{storage_account_fqdn}": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{storage_account_fqdn}": client_id,
    f"fs.azure.account.oauth2.client.secret.{storage_account_fqdn}": client_secret,
    f"fs.azure.account.oauth2.client.endpoint.{storage_account_fqdn}": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

# Mantido também em formato genérico, caso alguma célula posterior
# esteja usando esse padrão antigo de opções.
adls_options_generic = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": client_id,
    "fs.azure.account.oauth2.client.secret": client_secret,
    "fs.azure.account.oauth2.client.endpoint": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}


# ============================================================
# 1.4 PATHS DA TABELA
# ============================================================

tabela = "ecommerce_produtos"

path_raw = (
    f"abfss://{raw_container}@{storage_account}.dfs.core.windows.net/"
    f"{source_folder}/{tabela}.csv"
)

path_bronze = (
    f"abfss://{squad_container}@{storage_account}.dfs.core.windows.net/"
    f"bronze/{tabela}"
)


# ============================================================
# 1.5 PERÍODO DE REFERÊNCIA DA EXECUÇÃO
# ============================================================

_hoje = date.today()
ANO_EXEC = _hoje.year
MES_EXEC = _hoje.month
DIA_EXEC = _hoje.day


# ============================================================
# 1.6 LOG SEGURO DE VALIDAÇÃO
# ============================================================

print(f"Configuração finalizada para a tabela: {tabela}")
print(f"Período de execução: {DIA_EXEC}/{MES_EXEC}/{ANO_EXEC}")
print(f"Storage Account: {storage_account}")
print(f"Raw Container: {raw_container}")
print(f"Source Folder: {source_folder}")
print(f"Squad Container: {squad_container}")
print(f"Path Raw: {path_raw}")
print(f"Path Bronze: {path_bronze}")
print("Credenciais carregadas sem expor secrets.")

## 2. 🥉 Camada Bronze — Ingestão RAW → Bronze

Realiza a ingestão do arquivo CSV bruto da camada Raw para a camada Bronze, incluindo:

1. **Validação de contrato de dados**: verifica quantidade, nomes e ordem das colunas antes de qualquer transformação
2. **Metadados de auditoria**: `bronze_ingested_at`, `bronze_source_file`, `ano_particao`, `mes_particao`

| Coluna esperada | Tipo Bronze | Descrição |
|-----------------|-------------|-----------|
| `sku` | String | Código único do produto |
| `nome_produto` | String | Nome descritivo |
| `descricao` | String | Descrição detalhada |
| `id_categoria` | String | FK para tabela de categorias |
| `preco_lista` | String | Preço de lista (tratamento na Silver) |
| `unidade_medida` | String | Unidade de venda (un, kg, L, etc) |
| `nome_marca` | String | Marca do produto |
| `is_ativo` | String | Flag de ativação (tratamento na Silver) |

> 📌 O schema é lido como `string` (`inferSchema=false`) para preservar os dados exatamente como estão na fonte.

In [0]:
# ============================================================
# 2. EXTRAÇÃO E CARGA (RAW -> BRONZE)
# ============================================================
print(f"Lendo {tabela} da camada Raw...")

df_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .options(**adls_options)
    .load(path_raw)
)

print(f"Total de registros encontrados: {df_raw.count()}")

# Validação do contrato de dados (Schema Enforcement)
colunas_esperadas = [
    "sku", "nome_produto", "descricao", "id_categoria",
    "preco_lista", "unidade_medida", "nome_marca", "is_ativo"
]

colunas_atuais = df_raw.columns
colunas_faltantes = set(colunas_esperadas) - set(colunas_atuais)
colunas_extras    = set(colunas_atuais) - set(colunas_esperadas)

if colunas_faltantes or colunas_extras:
    raise ValueError(f"ERRO CRÍTICO: Faltam {list(colunas_faltantes)} / Sobram {list(colunas_extras)}")

# Mesmo com nomes e quantidade corretos, a ORDEM pode ter mudado na fonte
# (ex.: reordenação acidental do CSV) — comparação de listas é sensível à ordem
if colunas_atuais != colunas_esperadas:
    raise ValueError(
        f"ERRO CRÍTICO: Ordem das colunas diverge do contrato.\n"
        f"Esperado: {colunas_esperadas}\n"
        f"Recebido: {colunas_atuais}"
    )

print("Validação estrutural OK: quantidade, nomes e ordem das colunas corretos.")

# Metadados de auditoria e particionamento
timestamp_carga = current_timestamp()

df_bronze = (
    df_raw
    .withColumn("bronze_ingested_at", timestamp_carga)
    .withColumn("bronze_source_file", col("_metadata.file_path"))
    .withColumn("ano_particao", year(timestamp_carga))
    .withColumn("mes_particao", month(timestamp_carga))
)

print(f"Gravando fisicamente na camada Bronze: {path_bronze}")

(
    df_bronze.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .options(**adls_options)
    .partitionBy("ano_particao", "mes_particao")
    .save(path_bronze)
)

print("Ingestão Bronze finalizada com sucesso!")
display(df_bronze.limit(5))


## 3. 🥈 Camada Silver — Deduplicação, Tipagem Financeira e Normalização

Transformações aplicadas sobre a Bronze para garantir qualidade analítica:

### Regras Aplicadas

| # | Regra | Detalhe |
|---|-------|---------|
| 1 | **SKU único e não nulo** | `dropna(subset=["sku"])` + `dropDuplicates(["sku"])` |
| 2 | **Preço financeiro** | `regexp_replace(",", ".")` → `cast("decimal(10,2)")` → `filter(> 0)` |
| 3 | **Boolean normalizado** | `is_ativo` → `cast("boolean")` com `coalesce(lit(False))` para nulos |

### Schema de saída da Silver

| Coluna | Tipo | Descrição |
|--------|------|-----------|
| `sku` | String | Chave primária do produto |
| `nome_produto` | String | Nome descritivo |
| `descricao` | String | Descrição detalhada |
| `id_categoria` | String | FK para categorias |
| `preco_lista` | Decimal(10,2) | Preço normalizado (> 0) |
| `unidade_medida` | String | Unidade de venda |
| `nome_marca` | String | Marca |
| `is_ativo` | Boolean | Flag de ativação (sem nulos) |
| `silver_processed_at` | Timestamp | Auditoria |

### ⚠️ Idempotência da escrita (correção aplicada)

A Silver é recalculada a partir da **Bronze inteira** a cada execução (`spark.read...load(path_bronze)` lê todo o histórico acumulado, não só o lote do dia). Por isso a escrita usa **`mode("overwrite")`**, e não `append`.

Usar `append` aqui seria um bug: como o `dropDuplicates(["sku"])` deduplica apenas o conjunto em memória daquela execução, cada novo *run* voltaria a inserir uma cópia inteira do catálogo já deduplicado — multiplicando o `sku` na tabela física a cada execução e violando a regra "1 (PK) — `sku` não nulo e único na Silver" no nível da tabela persistida (e não só no DataFrame momentâneo).

In [0]:
# ============================================================
# 3. CAMADA SILVER (APLICAÇÃO DAS REGRAS TÉCNICAS)
# ============================================================

print(f"Iniciando processamento da camada Silver para: {tabela}...")

path_silver = f"abfss://squad3@{storage_account}.dfs.core.windows.net/silver/{tabela}"

# Leitura da Bronze
df_bronze_read = spark.read.format("delta").options(**adls_options).load(path_bronze)

df_silver = (
    df_bronze_read

    # REGRA 1: SKU não nulo e único
    .dropna(subset=["sku"])
    .dropDuplicates(["sku"])

    # Tipagens de texto
    .withColumn("sku", col("sku").cast("string"))
    .withColumn("nome_produto", col("nome_produto").cast("string"))
    .withColumn("descricao", col("descricao").cast("string"))
    .withColumn("unidade_medida", col("unidade_medida").cast("string"))
    .withColumn("nome_marca", col("nome_marca").cast("string"))
    .withColumn("id_categoria", col("id_categoria").cast("string"))

    # REGRA 2: preco_lista como Decimal(10,2) e > 0
    .withColumn("preco_lista", regexp_replace(col("preco_lista"), ",", ".").cast("decimal(10,2)"))
    .filter(col("preco_lista") > 0)

    # REGRA 3: is_ativo como boolean; nulos imputados como False
    .withColumn("is_ativo", coalesce(col("is_ativo").cast("boolean"), lit(False)))

    # Auditoria
    .withColumn("silver_processed_at", current_timestamp())
)

print(f"Gravando dados limpos em: {path_silver}")

# MODO: OVERWRITE (não append).
# df_silver é recalculado sobre a Bronze inteira a cada execução, já deduplicado
# por sku. Usar "append" faria a tabela física acumular cópias do catálogo a
# cada execução, voltando a duplicar o sku ao longo do tempo.
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .options(**adls_options)
    .partitionBy("ano_particao", "mes_particao")
    .save(path_silver)
)

print("SUCESSO! Regras aplicadas e dados gravados na Silver.\n")
display(df_silver.limit(5))


## 4. 🥇 Camada Gold — KPIs de Catálogo

Consome a Silver de `ecommerce_produtos` e gera dois KPIs batch.

### 4.1 KPI de Preço Médio Semestral por Subcategoria

Gera um **KPI de preço médio por categoria**, segmentado por semestre (S1: Jan–Jun, S2: Jul–Dez). Considera apenas produtos ativos (`is_ativo = True`).

### Estratégia de Cálculo

1. **Filtrar ativos**: apenas `is_ativo = True`
2. **Enriquecimento de categoria**: join com a Silver de `ecommerce_categorias` para trazer `nome_categoria` e classificar `tipo_categoria` (`Raiz` ou `Subcategoria`, a partir de `id_categoria_pai`)
3. **Semestre**: derivado do mês da execução (`<= 6` → S1, `> 6` → S2)
4. **Agregação**: `AVG(preco_lista)` agrupado por `id_categoria`, `nome_categoria`, `tipo_categoria`, `ano_particao`, `semestre_particao`
5. **Idempotência**: Delete + append com granularidade de dia de execução

### KPI Gerado

| Coluna | Tipo | Descrição |
|--------|------|-----------|
| `id_categoria` | String | FK para categoria |
| `nome_categoria` | String | Nome da categoria/subcategoria (via join com `ecommerce_categorias`) |
| `tipo_categoria` | String | `Raiz` ou `Subcategoria` |
| `ano_particao` | Integer | Ano (partição física) |
| `semestre_particao` | Integer | 1 ou 2 |
| `preco_medio_semestre` | Decimal | Preço médio dos produtos ativos |
| `dia_exec` | Integer | Dia da execução (controle de idempotência) |
| `gold_processed_at` | String | Timestamp de processamento |

> 📌 A escrita Gold utiliza **delete + append por dia de execução** (`dia_exec`), garantindo idempotência em reprocessamentos da mesma semana.

> ⚠️ **Requisito de negócio explícito**: "preço médio **por subcategoria**". Como nem todo produto necessariamente está cadastrado direto em uma subcategoria (alguns podem referenciar uma categoria raiz), a linha é mantida e classificada via `tipo_categoria` em vez de descartada silenciosamente — filtre por `tipo_categoria = 'Subcategoria'` no consumo (SQL/BI) se o requisito exigir exclusivamente subcategorias.

> 🔗 **Dependência cruzada**: este KPI lê a Silver de `ecommerce_categorias` (`silver/ecommerce_categorias`), que precisa já ter sido processada antes deste notebook.

In [0]:
# ============================================================
# 4.1 CAMADA GOLD (PRODUTOS - PREÇO MÉDIO SEMESTRAL POR SUBCATEGORIA)
# ============================================================

print("Iniciando processamento da camada Gold (Preço Médio por Semestre)...")

tabela_gold = "gold_hist_preco_medio_semestre"
path_gold_agg = f"abfss://squad3@{storage_account}.dfs.core.windows.net/gold/{tabela_gold}"

# Releitura da Silver de produtos
df_silver = spark.read.format("delta").options(**adls_options).load(path_silver)

# Leitura da Silver de categorias (dependência cruzada)
path_silver_categorias = f"abfss://squad3@{storage_account}.dfs.core.windows.net/silver/ecommerce_categorias"
df_categorias_full = spark.read.format("delta").options(**adls_options).load(path_silver_categorias)

df_categoria_info = (
    df_categorias_full
    .select(
        col("id_categoria").alias("id_categoria_lookup"),
        col("nome_categoria"),
        when(
            col("id_categoria_pai").isNull() | (col("id_categoria_pai") == "null"),
            lit("Raiz")
        ).otherwise(lit("Subcategoria")).alias("tipo_categoria")
    )
)

df_gold_agg = (
    df_silver
    .filter(col("is_ativo") == True)
    .join(df_categoria_info, col("id_categoria") == col("id_categoria_lookup").cast("double").cast("long").cast("string"), "left")

    .withColumn("data_fotografia", current_timestamp())
    .withColumn("ano_particao", year(col("data_fotografia")))
    .withColumn("mes", month(col("data_fotografia")))
    .withColumn("semestre_particao", when(col("mes") <= 6, 1).otherwise(2))

    .groupBy("id_categoria", "nome_categoria", "tipo_categoria", "ano_particao", "semestre_particao")
    .agg(
        round(avg("preco_lista"), 2).alias("preco_medio_semestre")
    )

    .withColumn("dia_exec", lit(DIA_EXEC))
    .withColumn("gold_processed_at", date_format(current_timestamp(), "yyyy-MM-dd HH:mm:ss"))
)

print("Gravando Data Mart na camada Gold (MODO: DELETE + APPEND)...")

# Delete + Append com granularidade de dia
try:
    (
        spark.read
        .format("delta")
        .options(**adls_options)
        .load(path_gold_agg)
        .createOrReplaceTempView(f"tmp_{tabela_gold}")
    )
    spark.sql(f"""
        DELETE FROM delta.`{path_gold_agg}`
        WHERE ano_particao       = {ANO_EXEC}
          AND semestre_particao   = {1 if MES_EXEC <= 6 else 2}
          AND dia_exec           = {DIA_EXEC}
    """)
    print(f"Delete executado: {tabela_gold} [{DIA_EXEC}/{MES_EXEC}/{ANO_EXEC}]")
except Exception:
    print(f"Primeira carga detectada — sem delete: {tabela_gold}")

# Append
(
    df_gold_agg.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .options(**adls_options)
    .partitionBy("ano_particao", "semestre_particao")
    .save(path_gold_agg)
)

print("SUCESSO! Preço médio semestral gravado na Gold.\n")
display(df_gold_agg.orderBy("ano_particao", "semestre_particao", "nome_categoria"))


### 4.2 KPI de SKUs Ativos por Categoria por Mês

Gera um **KPI de contagem de SKUs ativos**, agrupado por categoria e por mês — usado para monitorar o crescimento do catálogo ao longo do tempo.

### Estratégia de Cálculo

1. **Filtrar ativos**: apenas `is_ativo = True`
2. **Enriquecimento de categoria**: reaproveita o mesmo `df_categoria_info` (join com `ecommerce_categorias`) usado no KPI 4.1
3. **Contagem distinta**: `COUNT(DISTINCT sku)` agrupado por `id_categoria`, `nome_categoria`, `tipo_categoria`, `ano_particao`, `mes_particao` — `countDistinct` é usado deliberadamente (em vez de `count`) para blindar o KPI contra qualquer duplicidade residual de `sku` que ainda exista na Silver
4. **Idempotência**: Delete + append com granularidade de dia de execução

### KPI Gerado

| Coluna | Tipo | Descrição |
|--------|------|-----------|
| `id_categoria` | String | FK para categoria |
| `nome_categoria` | String | Nome da categoria/subcategoria |
| `tipo_categoria` | String | `Raiz` ou `Subcategoria` |
| `ano_particao` | Integer | Ano (partição física) |
| `mes_particao` | Integer | Mês (partição física) |
| `skus_ativos` | Integer | Quantidade de SKUs ativos distintos na categoria/mês |
| `dia_exec` | Integer | Dia da execução (controle de idempotência) |
| `gold_processed_at` | String | Timestamp de processamento |

> 📌 A escrita Gold utiliza **delete + append por dia de execução** (`dia_exec`), seguindo o mesmo padrão do KPI 4.1.

In [0]:
# ============================================================
# 4.2 CAMADA GOLD (PRODUTOS - SKUS ATIVOS POR CATEGORIA POR MÊS)
# ============================================================

print("Iniciando processamento da camada Gold (SKUs Ativos por Categoria/Mês)...")

tabela_gold_skus = "gold_kpi_skus_ativos_categoria_mes"
path_gold_skus = f"abfss://squad3@{storage_account}.dfs.core.windows.net/gold/{tabela_gold_skus}"

df_gold_skus_categoria_mes = (
    df_silver
    .filter(col("is_ativo") == True)
    .join(df_categoria_info, col("id_categoria") == col("id_categoria_lookup").cast("double").cast("long").cast("string"), "left")

    .withColumn("data_fotografia", current_timestamp())
    .withColumn("ano_particao", year(col("data_fotografia")))
    .withColumn("mes_particao", month(col("data_fotografia")))

    .groupBy("id_categoria", "nome_categoria", "tipo_categoria", "ano_particao", "mes_particao")
    .agg(
        countDistinct("sku").alias("skus_ativos")
    )

    .withColumn("dia_exec", lit(DIA_EXEC))
    .withColumn("gold_processed_at", date_format(current_timestamp(), "yyyy-MM-dd HH:mm:ss"))
)

print("Gravando Data Mart na camada Gold (MODO: DELETE + APPEND)...")

# Delete + Append com granularidade de dia
try:
    (
        spark.read
        .format("delta")
        .options(**adls_options)
        .load(path_gold_skus)
        .createOrReplaceTempView(f"tmp_{tabela_gold_skus}")
    )
    spark.sql(f"""
        DELETE FROM delta.`{path_gold_skus}`
        WHERE ano_particao = {ANO_EXEC}
          AND mes_particao  = {MES_EXEC}
          AND dia_exec      = {DIA_EXEC}
    """)
    print(f"Delete executado: {tabela_gold_skus} [{DIA_EXEC}/{MES_EXEC}/{ANO_EXEC}]")
except Exception:
    print(f"Primeira carga detectada — sem delete: {tabela_gold_skus}")

# Append
(
    df_gold_skus_categoria_mes.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .options(**adls_options)
    .partitionBy("ano_particao", "mes_particao")
    .save(path_gold_skus)
)

print("SUCESSO! SKUs ativos por categoria/mês gravado na Gold.\n")
display(df_gold_skus_categoria_mes.orderBy("ano_particao", "mes_particao", "nome_categoria"))


## 5. 🗄️ Exportação para SQL Server

Exporta os dois KPIs Gold para o SQL Server no schema `squad3`, modo `append` (ambos usam delete + append por `dia_exec` na Gold, então o append no SQL Server replica o mesmo fragmento incremental).

| Tabela SQL Server | Data Mart de origem |
|-------------------|---------------------|
| `squad3.gold_hist_preco_medio_semestre` | `df_gold_agg` |
| `squad3.gold_kpi_skus_ativos_categoria_mes` | `df_gold_skus_categoria_mes` |

> ⚠️ Credenciais carregadas via `load_dotenv()` — nunca expostas no código.

In [0]:
# ============================================================
# 5. EXPORTAÇÃO PARA O SQL SERVER (SERVING LAYER)
# ============================================================
import os
from dotenv import load_dotenv

print("Iniciando a exportação dos KPIs para o SQL Server (MODO APPEND)...")

load_dotenv("../env")

jdbc_hostname = os.getenv("SQL_HOST")
jdbc_port     = "1433"
jdbc_database = os.getenv("SQL_DATABASE")
jdbc_username = os.getenv("SQL_USERNAME")
jdbc_password = os.getenv("SQL_PASSWORD")


def carregar_no_sql_server(df, tabela_sql, mode="append"):
    """
    Grava um DataFrame PySpark em uma tabela do SQL Server via conector nativo,
    isolando falhas por tabela (uma exportação não derruba as demais).
    """
    try:
        # Reprojeta as colunas (mesmos nomes, plano "limpo") antes do write.
        # Sem isso, DataFrames vindos de join+groupBy às vezes disparam
        # COLUMN_NOT_DEFINED_IN_TABLE no conector JDBC mesmo quando a coluna
        # existe nos dois lados — resíduo de atributo da linhagem do join.
        df_export = df.select(*df.columns)
        (
            df_export.write
            .format("sqlserver")
            .option("host", jdbc_hostname)
            .option("port", jdbc_port)
            .option("database", jdbc_database)
            .option("dbtable", tabela_sql)
            .option("user", jdbc_username)
            .option("password", jdbc_password)
            .mode(mode)
            .save()
        )
        print(f"SUCESSO! Dados exportados para {tabela_sql} no SQL Server.")
    except Exception as e:
        print(f"Erro ao exportar para {tabela_sql}:\n{e}")


carregar_no_sql_server(df_gold_agg, "squad3.gold_hist_preco_medio_semestre", mode="append")
carregar_no_sql_server(df_gold_skus_categoria_mes, "squad3.gold_kpi_skus_ativos_categoria_mes", mode="append")


## 6. 📊 Data Quality — Sanidade Analítica (Catálogo de Produtos)

Análise gráfica e textual da qualidade dos dados em cada etapa de transformação, monitorando os 7 problemas mais comuns na validação de produtos — incluindo a unicidade de `sku` na Silver **persistida** (R6) e a integridade do novo KPI de SKUs ativos (R7).

In [0]:
# ============================================================
# 6. DATA QUALITY - MONITORAMENTO DO CATÁLOGO DE PRODUTOS
# ============================================================
from pyspark.sql.functions import col as col_func
from builtins import round as python_round
import matplotlib.pyplot as plt
import pandas as pd

print("Gerando insights de qualidade da Silver (Produtos)...\n")

# Releitura da Bronze e Silver
df_bronze_check = spark.read.format("delta").options(**adls_options).load(path_bronze)
df_silver_check = spark.read.format("delta").options(**adls_options).load(path_silver)
total_registros_bronze = df_bronze_check.count()
total_registros_silver = df_silver_check.count()

# REGRA 1: SKU nulo (Bronze)
regra_1_erros = df_bronze_check.filter(col_func("sku").isNull()).count()

# REGRA 2: SKU duplicado (Bronze)
regra_2_erros = total_registros_bronze - df_bronze_check.dropDuplicates(["sku"]).count()

# REGRA 3: preco_lista inválido (Silver — nulos ou <= 0 após cast)
df_preco_check = (
    df_bronze_check
    .withColumn("preco_cast", regexp_replace(col_func("preco_lista"), ",", ".").cast("decimal(10,2)"))
)
regra_3_erros = df_preco_check.filter(
    col_func("preco_cast").isNull() | (col_func("preco_cast") <= 0)
).count()

# REGRA 4: is_ativo nulo na Bronze (imputados como False na Silver)
regra_4_erros = df_bronze_check.filter(col_func("is_ativo").isNull()).count()

# REGRA 5: Produtos inativos residuais na Silver (is_ativo = False)
regra_5_erros = df_silver_check.filter(col_func("is_ativo") == False).count()

# REGRA 6: SKU duplicado na Silver PERSISTIDA (tabela física) — prova de que a
# escrita em overwrite eliminou a duplicação entre execuções (bug anterior do
# mode("append") sobre um DataFrame recalculado a cada run)
regra_6_erros = total_registros_silver - df_silver_check.select("sku").distinct().count()

# REGRA 7: id_categoria sem correspondência em ecommerce_categorias (nome_categoria
# nulo após o join) no KPI de SKUs ativos — detecta FK quebrada ou categoria
# cadastrada em produtos mas ausente na dimensão de categorias
total_skus_ativos_kpi = df_gold_skus_categoria_mes.count()
regra_7_erros = df_gold_skus_categoria_mes.filter(col_func("nome_categoria").isNull()).count()

# ============================================================
# Montar DataFrame com os resultados
# ============================================================

dados_qualidade = {
    'Regra': [
        'R1: SKU\nNULO (Bronze)',
        'R2: SKU\nDuplicado (Bronze)',
        'R3: Preço\nInválido (Bronze)',
        'R4: is_ativo\nNULO (Bronze)',
        'R5: Produtos\nInativos (Silver)',
        'R6: SKU\nDuplicado (Silver)',
        'R7: Categoria\nsem match (Gold)'
    ],
    'Erros Encontrados': [
        regra_1_erros,
        regra_2_erros,
        regra_3_erros,
        regra_4_erros,
        regra_5_erros,
        regra_6_erros,
        regra_7_erros
    ],
    'Taxa (%)': [
        python_round(100 * regra_1_erros / total_registros_bronze, 2) if total_registros_bronze > 0 else 0,
        python_round(100 * regra_2_erros / total_registros_bronze, 2) if total_registros_bronze > 0 else 0,
        python_round(100 * regra_3_erros / total_registros_bronze, 2) if total_registros_bronze > 0 else 0,
        python_round(100 * regra_4_erros / total_registros_bronze, 2) if total_registros_bronze > 0 else 0,
        python_round(100 * regra_5_erros / total_registros_silver, 2) if total_registros_silver > 0 else 0,
        python_round(100 * regra_6_erros / total_registros_silver, 2) if total_registros_silver > 0 else 0,
        python_round(100 * regra_7_erros / total_skus_ativos_kpi, 2) if total_skus_ativos_kpi > 0 else 0,
    ]
}

df_qualidade = pd.DataFrame(dados_qualidade)

# ============================================================
# Exibir Tabela Resumo
# ============================================================

print(f"\n{'='*70}")
print(f"ANÁLISE DE QUALIDADE - REGRAS SILVER/GOLD (PRODUTOS)")
print(f"{'='*70}")
print(f"Total de registros na Bronze: {total_registros_bronze:,}")
print(f"Total de registros na Silver: {total_registros_silver:,}")
print(f"Duplicatas removidas: {total_registros_bronze - total_registros_silver:,}")
print(f"Taxa de rejeição geral: {python_round(100 * (total_registros_bronze - total_registros_silver) / total_registros_bronze, 2)}%")
print(f"Total de linhas no KPI de SKUs ativos por categoria/mês (Gold): {total_skus_ativos_kpi:,}\n")
print(df_qualidade.to_string(index=False))
print(f"{'='*70}\n")

# ============================================================
# Gráficos
# ============================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colors_gradient = ['#ff6b6b', '#ff7e7e', '#ff9191', '#ffa4a4', '#ffb7b7', '#ffcaca', '#ffdddd']
bars1 = ax1.bar(range(len(df_qualidade)), df_qualidade['Erros Encontrados'], color=colors_gradient, edgecolor='#333', linewidth=1.5)
ax1.set_ylabel('Quantidade de Registros com Erro', fontsize=11, fontweight='bold')
ax1.set_title('Erros Detectados por Regra de Validação', fontsize=12, fontweight='bold')
ax1.set_xticks(range(len(df_qualidade)))
ax1.set_xticklabels(df_qualidade['Regra'], fontsize=9)
ax1.grid(axis='y', alpha=0.3, linestyle='--')

for bar in bars1:
    height = bar.get_height()
    if height > 0:
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(height):,}',
                ha='center', va='bottom', fontsize=9, fontweight='bold')

colors_taxa = ['#e74c3c' if x > 5 else '#f39c12' if x > 1 else '#27ae60' for x in df_qualidade['Taxa (%)']]
bars2 = ax2.bar(range(len(df_qualidade)), df_qualidade['Taxa (%)'], color=colors_taxa, edgecolor='#333', linewidth=1.5)
ax2.set_ylabel('Taxa de Erro (%)', fontsize=11, fontweight='bold')
ax2.set_title('Taxa de Erro por Regra (% do Total)', fontsize=12, fontweight='bold')
ax2.set_xticks(range(len(df_qualidade)))
ax2.set_xticklabels(df_qualidade['Regra'], fontsize=9)
ax2.grid(axis='y', alpha=0.3, linestyle='--')
ax2.axhline(y=1, color='#f39c12', linestyle='--', linewidth=1, alpha=0.5, label='Limite Aceitável (1%)')

for bar in bars2:
    height = bar.get_height()
    if height > 0:
        ax2.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.2f}%',
                ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

print("✅ Insights de qualidade gerados com sucesso.\n")


In [0]:
# # ============================================================
# # TRUNCATE (executar UMA VEZ para limpar duplicatas acumuladas)
# # ============================================================

# path_bronze_trunc = f"abfss://squad3@{storage_account}.dfs.core.windows.net/bronze/{tabela}"
# path_silver_trunc = f"abfss://squad3@{storage_account}.dfs.core.windows.net/silver/{tabela}"
# path_gold_trunc   = f"abfss://squad3@{storage_account}.dfs.core.windows.net/gold/gold_hist_preco_medio_semestre"

# for path in [path_bronze_trunc, path_silver_trunc, path_gold_trunc]:
#     try:
#         df_vazio = spark.read.format("delta").options(**adls_options).load(path).limit(0)
#         df_vazio.write.format("delta").mode("overwrite").options(**adls_options).save(path)
#         print(f"✅ Truncado: {path.split('/')[-1]}")
#     except Exception:
#         print(f"⏭️ Tabela não encontrada (skip): {path.split('/')[-1]}")

# print("\nPronto! Re-execute o pipeline do início.")
